In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import shutil
from pathlib import Path
import os
import glob
import math
from PIL import Image
import numpy as np
from tqdm import tqdm
from collections import Counter, defaultdict
import random

# Wczytanie i rozpakowanie danych

In [3]:
path_drive_data_zip = Path(
    '/content/drive/MyDrive/dane_detekcja_pojazdow/EAGLE_Dataset_public_edit_slicing_more.zip'
    )

path_data_zip = Path('/content/data.zip')

shutil.copy(path_drive_data_zip, path_data_zip)

PosixPath('/content/data.zip')

In [4]:
!unzip -q /content/data.zip -d /content/data

In [7]:
data_path = Path('/content/data/content/EAGLE_Dataset_public_edit_slicing_more')

Wypisanie liczby danych

In [8]:
# liczba plików
def count_files(data_path = data_path):
  for subforlder in ['val', 'train', 'test']:
    print(subforlder)
    img_num = len(list((data_path / 'images' / subforlder).glob('*')))
    label_num = len(list((data_path / 'labels' / subforlder).glob('*')))

    img_background_num = (img_num - label_num  ) / img_num

    print("Obrazy:" ,img_num)
    print("Etykiety:", label_num)
    print(f"Obrazy tła: {img_background_num*100:.2f}%")
    print("")

count_files()


val
Obrazy: 9753
Etykiety: 6314
Obrazy tła: 35.26%

train
Obrazy: 29611
Etykiety: 19368
Obrazy tła: 34.59%

test
Obrazy: 3661
Etykiety: 2920
Obrazy tła: 20.24%



# Wczytanie wymiarów obrazów testowych

In [9]:
def get_dim_from_image_path(file_path):
  """
  zwraca width i height obrazu z podanego path
  """
  with Image.open(file_path) as img:
      return img.size


In [10]:
test_images_path = Path("/content/data/content/EAGLE_Dataset_public_edit_slicing_more/images/test")

test_images_dimensions = {}

for file_path in test_images_path.glob('*'):
  dimensions = get_dim_from_image_path(file_path)
  test_images_dimensions[file_path.stem] = {
      'w': dimensions[0],
      'h': dimensions[1]
      }

# pierwsze 5 rekordów
dict(list(test_images_dimensions.items())[:5])


{'2017-07-19-Benchmark-Flug_1-L-L4623__1024__1536___768': {'w': 1024,
  'h': 1024},
 '2018-10-04-Ampelkreuzungen_L0017__1024__768___2432': {'w': 1024, 'h': 1024},
 '2018-10-26-Freiham-A96-original-mitte_4K0G0227__1024__0___2720': {'w': 1024,
  'h': 1024},
 '2014-07-25-Chicago--links-F0019__1024__4592___768': {'w': 1024, 'h': 1024},
 '2016-11-15-A8-A96-A7-München-satz1-L-L0131__1024__1536___2432': {'w': 1024,
  'h': 1024}}

# Filtracja

In [12]:
def get_image_dimensions(filename_stem, split):
    """
    Zwraca (width, height) dla danego pliku w konkretnym zbiorze.
    Args:
        filename_stem (str): nazwa pliku bez rozszerzenia (np. 'P001_rate1.0')
        split (str): 'train', 'val' lub 'test'
    """

    dim = str(filename_stem).split('__')[-3]
    return int(dim), int(dim)

print(get_image_dimensions('2016-11-15-A8-A96-A7-München-satz1-L-L0131__1024__1536___2432.jpg', 'test'))


(1024, 1024)


In [17]:
# Parametry filtracji
IMG_GSD_THRESHOLD_PX = 18.0  # Jeśli mediana < 18px -> USUŃ PLIK
SINGLE_OBJ_MIN_PX = 12.0     # Jeśli obiekt < 12px -> usuń obiekt

DRY_RUN = False  # True = Tylko wypisz co byś zrobił. False = KASUJ DANE.

In [15]:
def get_paths(base_dir, split):
    """Zwraca ścieżki do labels i images dla danego splitu."""
    # Dostosuj, jeśli masz inną strukturę katalogów
    lbl_path = base_dir / 'labels' / split
    img_path = base_dir / 'images' / split
    return lbl_path, img_path


def calculate_obb_long_edge_rect(coords, img_w, img_h):
    """Oblicza dłuższy bok OBB w pikselach (denormalizacja)."""
    x_coords = coords[0::2]
    y_coords = coords[1::2]

    px_x = [x * img_w for x in x_coords]
    px_y = [y * img_h for y in y_coords]

    p1 = (px_x[0], px_y[0])
    p2 = (px_x[1], px_y[1])
    p3 = (px_x[2], px_y[2])

    edge1 = math.hypot(p1[0] - p2[0], p1[1] - p2[1])
    edge2 = math.hypot(p2[0] - p3[0], p2[1] - p3[1])

    return max(edge1, edge2)

def remove_associated_image(img_dir, stem):
    """Usuwa obraz sprawdzając .jpg i .JPG"""
    for ext in ['.jpg', '.JPG']:
        img_path = os.path.join(img_dir, stem + ext)
        if os.path.exists(img_path):
            if not DRY_RUN:
                os.remove(img_path)
            return True # Znaleziono i usunięto (lub symulowano)
    return False # Nie znaleziono pliku obrazu

def process_split(split):
    lbl_dir, img_dir = get_paths(data_path, split)

    if not os.path.exists(lbl_dir):
        print(f"[SKIP] Nie znaleziono folderu: {lbl_dir}")
        return

    txt_files = glob.glob(os.path.join(lbl_dir, '*.txt'))

    stats = {
        'split': split,
        'total': len(txt_files),
        'removed_files': 0,
        'removed_objects': 0,
        'kept_objects': 0
    }

    print(f"\n--- Przetwarzanie: {split.upper()} ({len(txt_files)} plików) ---")

    for txt_path in tqdm(txt_files):
        filename = os.path.basename(txt_path)
        stem = os.path.splitext(filename)[0]

        # 1. Pobierz wymiary (z parametrem split!)
        try:
            img_w, img_h = get_image_dimensions(stem, split)
        except Exception as e:
            print(f"[BŁĄD] Brak wymiarów dla {split}/{filename}: {e}")
            continue

        # 2. Wczytaj etykiety
        with open(txt_path, 'r') as f:
            lines = f.readlines()

        if not lines:
            # Pusty plik -> opcjonalnie usuń. Tutaj zakładam usuwanie.
            remove_associated_image(img_dir, stem)
            if not DRY_RUN: os.remove(txt_path)
            stats['removed_files'] += 1
            continue

        objects = []
        for line in lines:
            parts = list(map(float, line.strip().split()))
            # Format YOLO OBB: class x1 y1 x2 y2 x3 y3 x4 y4
            coords = parts[1:]
            size_px = calculate_obb_long_edge_rect(coords, img_w, img_h)
            objects.append({'line': line, 'size': size_px})

        if not objects:
            continue

        # 3. Logika filtracji
        sizes = [o['size'] for o in objects]
        median_size = np.median(sizes)

        # A. Usuwanie całego pliku (słabe GSD)
        if median_size < IMG_GSD_THRESHOLD_PX:
            stats['removed_files'] += 1
            # Usuń obraz (obsługa jpg/JPG)
            remove_associated_image(img_dir, stem)
            # Usuń txt
            if not DRY_RUN:
                os.remove(txt_path)

        # B. Filtrowanie pojedynczych obiektów
        else:
            new_lines = []
            file_dirty = False

            for obj in objects:
                if obj['size'] < SINGLE_OBJ_MIN_PX:
                    stats['removed_objects'] += 1
                    file_dirty = True
                else:
                    new_lines.append(obj['line'])
                    stats['kept_objects'] += 1

            # Zapis zmian
            if file_dirty and not DRY_RUN:
                if not new_lines:
                    # Jeśli po czyszczeniu nic nie zostało -> usuń plik i obraz
                    remove_associated_image(img_dir, stem)
                    os.remove(txt_path)
                    stats['removed_files'] += 1
                else:
                    # Nadpisz tylko txt
                    with open(txt_path, 'w') as f:
                        f.writelines(new_lines)
            elif not file_dirty:
                stats['kept_objects'] += len(objects)

    return stats

In [18]:
splits = ['train', 'val', 'test'] # Lista zbiorów do przeliterowania

print(f"START SKRYPTU. Tryb DRY_RUN = {DRY_RUN}")

total_stats = []
for split in splits:
    s = process_split(split)
    if s: total_stats.append(s)

print("\n" + "="*40)
print("PODSUMOWANIE KOŃCOWE")
print("="*40)
for s in total_stats:
    print(f"ZBIÓR: {s['split'].upper()}")
    print(f"  - Pliki usunięte całkowicie: {s['removed_files']}")
    print(f"  - Pojedyncze obiekty usunięte: {s['removed_objects']}")
    print(f"  - Obiekty zachowane: {s['kept_objects']}")
    print("-" * 20)

START SKRYPTU. Tryb DRY_RUN = False

--- Przetwarzanie: TRAIN (19368 plików) ---


100%|██████████| 19368/19368 [00:06<00:00, 3118.21it/s]



--- Przetwarzanie: VAL (6314 plików) ---


100%|██████████| 6314/6314 [00:01<00:00, 3336.86it/s]



--- Przetwarzanie: TEST (2920 plików) ---


100%|██████████| 2920/2920 [00:01<00:00, 2846.74it/s]


PODSUMOWANIE KOŃCOWE
ZBIÓR: TRAIN
  - Pliki usunięte całkowicie: 1347
  - Pojedyncze obiekty usunięte: 10729
  - Obiekty zachowane: 837046
--------------------
ZBIÓR: VAL
  - Pliki usunięte całkowicie: 608
  - Pojedyncze obiekty usunięte: 2663
  - Obiekty zachowane: 258012
--------------------
ZBIÓR: TEST
  - Pliki usunięte całkowicie: 152
  - Pojedyncze obiekty usunięte: 1438
  - Obiekty zachowane: 180795
--------------------


In [19]:
count_files()

val
Obrazy: 9145
Etykiety: 5706
Obrazy tła: 37.61%

train
Obrazy: 28264
Etykiety: 18021
Obrazy tła: 36.24%

test
Obrazy: 3509
Etykiety: 2768
Obrazy tła: 21.12%



# Usunięcie nadmiarowych obrazów tła
(powinno być 10%)

In [20]:
background_val_images = set()
background_train_images = set()

# Rozszerzenia obrazów do sprawdzenia (żeby nie łapać śmieci)
valid_extensions = {'.jpg', '.JPG'}

for split_ in ["val", "train"]:
    print(f"Przetwarzanie {split_}...")
    images_path = data_path / "images" / split_
    labels_path = data_path / "labels" / split_

    # 1. Pobierz wszystkie obrazy raz
    # Używamy rglob lub glob, ale tylko raz na folder
    image_files = [f for f in images_path.glob("*") if f.suffix.lower() in valid_extensions]

    for img_file in image_files:
        # Konstruujemy potencjalną ścieżkę do etykiety
        label_file = labels_path / f"{img_file.stem}.txt"

        is_background = False

        # Przypadek A: Plik etykiety w ogóle nie istnieje
        if not label_file.exists():
            is_background = True

        # Przypadek B: Plik istnieje, ale jest pusty (częste przy kafelkowaniu)
        else:
            # stat().st_size == 0 sprawdza rozmiar pliku w bajtach (bardzo szybkie)
            if label_file.stat().st_size == 0:
                is_background = True

        if is_background:
            if split_ == "val":
                background_val_images.add(img_file.stem)
            else:
                background_train_images.add(img_file.stem)

print(f"Liczba teł w Val: {len(background_val_images)}")
print(f"Liczba teł w Train: {len(background_train_images)}")



Przetwarzanie val...
Przetwarzanie train...
Liczba teł w Val: 3439
Liczba teł w Train: 10243


In [21]:
# Funkcja pomocnicza, żeby nie pisać tego samego kodu dwa razy
def count_backgrounds_by_parent(tile_names_set):
    # Tworzymy licznik
    parent_counts = Counter()

    for tile_name in tile_names_set:
        # Wyciągamy nazwę rodzica (to co przed pierwszym podwójnym podkreśleniem)
        parent_name = tile_name.split("__")[0]
        parent_counts[parent_name] += 1

    return parent_counts

# 1. Tworzenie struktur licznikowych
train_bg_counts = count_backgrounds_by_parent(background_train_images)
val_bg_counts = count_backgrounds_by_parent(background_val_images)

# --- Wyświetlanie wyników ---

print(f"--- TRAIN ({len(train_bg_counts)} unikalnych obrazów rodziców) ---")
# most_common(5) pokaże 5 obrazów, które wygenerowały najwięcej tła
for parent, count in train_bg_counts.most_common(15):
    print(f"{parent}: {count} kafelków tła")

print(f"\n--- VAL ({len(val_bg_counts)} unikalnych obrazów rodziców) ---")
for parent, count in val_bg_counts.most_common(15):
    print(f"{parent}: {count} kafelków tła")

--- TRAIN (159 unikalnych obrazów rodziców) ---
2017-04-20-Braunschweig-Frankfurt--mitte-4K0G0084: 184 kafelków tła
2015-03-25-Elmau-mitte-4K0G0446: 182 kafelków tła
2017-04-20-Braunschweig-Frankfurt--mitte-4K0G0824: 181 kafelków tła
2017-04-20-Braunschweig-Frankfurt--mitte-4K0G0330: 181 kafelków tła
2017-04-20-Braunschweig-Frankfurt--mitte-4K0G0434: 180 kafelków tła
2017-07-03-Braunschweig-set1-N-4K0G0579: 166 kafelków tła
2017-04-20-Braunschweig-Frankfurt--mitte-4K0G0034: 166 kafelków tła
2016-02-PHAROS_SOLOSNA-2016-03-01-Solsona-rechts-R0079: 163 kafelków tła
2016-02-PHAROS_SOLOSNA-2016-03-02-Solsona-rechts-R0011: 160 kafelków tła
2016-11-15-A8-A96-A7-München-satz1-R-R6120: 157 kafelków tła
2012-05-19-Champions-League-Final-rechts-R0275: 155 kafelków tła
2018-10-26-Freiham-A96-original-mitte_4K0G0008: 154 kafelków tła
2012-04-26-Muenchen-Tunnel_4K0G0030: 151 kafelków tła
2014-07-25-Chicago--mitte-S0625: 149 kafelków tła
2015-04-09-OP-mitte-4K0G0327: 149 kafelków tła

--- VAL (53 un

In [22]:
def get_files_to_remove(background_set, images_path, target_percentage=0.10):
    """
    background_set: zbiór nazw plików (stems) które są tłem
    images_path: ścieżka do folderu z obrazami (żeby policzyć ile jest FG)
    target_percentage: cel w ułamku (0.10 = 10%)
    """

    # 1. Grupowanie kafelków tła po rodzicu
    # Zamieniamy płaski zbiór {'img_0_0', 'img_0_1'} na {'img': ['img_0_0', 'img_0_1']}
    parent_groups = defaultdict(list)
    for tile_name in background_set:
        parent_name = tile_name.split("__")[0]
        parent_groups[parent_name].append(tile_name)

    # 2. Matematyka zbioru
    # Musimy wiedzieć ile mamy wszystkich plików, żeby obliczyć Foreground
    # (używamy szybkiego listowania generatora)
    all_files_count = sum(1 for _ in images_path.glob('*') if _.suffix in {'.jpg', '.png', '.jpeg', '.tif'})
    current_bg_count = len(background_set)
    foreground_count = all_files_count - current_bg_count

    print(f"--- Statystyki dla {images_path.name} ---")
    print(f"Wszystkich plików: {all_files_count}")
    print(f"Obiekty (FG): {foreground_count}")
    print(f"Obecne tło (BG): {current_bg_count} ({current_bg_count/all_files_count:.1%})")

    # Obliczamy cel
    # Wzór: Target_BG / (FG + Target_BG) = target_percentage
    # Przekształcenie: Target_BG = FG * (target_percentage / (1 - target_percentage))
    # Dla 10%: Target_BG = FG * (0.1 / 0.9) = FG / 9

    if target_percentage >= 1.0:
        print("Błąd: Procent musi być ułamkiem < 1.0")
        return []

    target_bg_count = int(foreground_count * (target_percentage / (1 - target_percentage)))

    print(f"Cel tła (BG): {target_bg_count} (by osiągnąć {target_percentage:.0%})")

    if target_bg_count >= current_bg_count:
        print("Obecna ilość tła jest już mniejsza lub równa celowi. Nic do usunięcia.")
        return []

    # Współczynnik redukcji - jaką część obecnego tła musimy zostawić?
    global_keep_ratio = target_bg_count / current_bg_count
    print(f"Współczynnik zachowania (Global Keep Ratio): {global_keep_ratio:.4f}")

    files_to_remove = []

    # 3. Losowanie per rodzic
    for parent, tiles in parent_groups.items():
        n_total_parent = len(tiles)

        # Ile zostawiamy z tego konkretnego rodzica?
        n_keep = int(n_total_parent * global_keep_ratio)

        # Opcjonalnie: Zawsze zostaw chociaż 1 kafelek tła z danego rodzica, jeśli to możliwe,
        # żeby zachować różnorodność scenerii (chyba że n_keep wyjdzie 0 przy bardzo ostrym cięciu)
        if n_keep == 0 and n_total_parent > 0 and target_bg_count > 0:
             # To jest moment decyzji: czy wolisz idealne 10% (wtedy zostaw 0),
             # czy różnorodność (wtedy wymuś 1).
             # Przy 180 kafelkach na obraz n_keep raczej nie będzie zerem, więc to edge case.
             pass

        # Mieszamy listę kafelków tego rodzica, żeby losowo wybrać te do usunięcia
        random.shuffle(tiles)

        # Dodajemy do listy usunięcia te, które są "nad kreską"
        # tiles[n_keep:] to wszystko powyżej liczby n_keep
        files_to_remove.extend(tiles[n_keep:])

    print(f"Zaplanowano do usunięcia: {len(files_to_remove)} plików.\n")
    return files_to_remove

# --- UŻYCIE ---

# Definiujemy ścieżki (zakładam że zmienna data_path istnieje z poprzedniego kodu)
train_img_path = data_path / "images" / "train"
val_img_path = data_path / "images" / "val"

# Generujemy listy (na razie nic nie usuwamy, tylko zbieramy nazwy)
remove_train = get_files_to_remove(background_train_images, train_img_path, target_percentage=0.10)
remove_val = get_files_to_remove(background_val_images, val_img_path, target_percentage=0.10)

# Przykład co wyleci
print("Przykładowe pliki do usunięcia z Train:", remove_train[:3])

--- Statystyki dla train ---
Wszystkich plików: 28264
Obiekty (FG): 18021
Obecne tło (BG): 10243 (36.2%)
Cel tła (BG): 2002 (by osiągnąć 10%)
Współczynnik zachowania (Global Keep Ratio): 0.1955
Zaplanowano do usunięcia: 8316 plików.

--- Statystyki dla val ---
Wszystkich plików: 9145
Obiekty (FG): 5706
Obecne tło (BG): 3439 (37.6%)
Cel tła (BG): 634 (by osiągnąć 10%)
Współczynnik zachowania (Global Keep Ratio): 0.1844
Zaplanowano do usunięcia: 2831 plików.

Przykładowe pliki do usunięcia z Train: ['2013-06-07-Crowd-rar2013_gsd_090_4K0G0335__512__384___2304', '2013-06-07-Crowd-rar2013_gsd_090_4K0G0335__512__4224___2688', '2013-06-07-Crowd-rar2013_gsd_090_4K0G0335__512__5104___384']


In [23]:
def delete_files(file_stems, images_dir, labels_dir):
    count = 0
    for stem in file_stems:
        # Szukamy pliku z dowolnym rozszerzeniem obrazu
        # (zakładam, że stem jest unikalny w folderze)
        found_img = list(images_dir.glob(f"{stem}.*"))
        if found_img:
            img_path = found_img[0]
            txt_path = labels_dir / f"{stem}.txt"

            try:
                os.remove(img_path) # Usuń obraz
                if txt_path.exists():
                    os.remove(txt_path) # Usuń pusty txt jeśli jest
                count += 1
            except OSError as e:
                print(f"Błąd przy {stem}: {e}")

    print(f"Usunięto fizycznie {count} par plików z {images_dir.name}.")


delete_files(remove_train, data_path / "images" / "train", data_path / "labels" / "train")
delete_files(remove_val, data_path / "images" / "val", data_path / "labels" / "val")

Usunięto fizycznie 8316 par plików z train.
Usunięto fizycznie 2831 par plików z val.


In [24]:
count_files()

val
Obrazy: 6314
Etykiety: 5706
Obrazy tła: 9.63%

train
Obrazy: 19948
Etykiety: 18021
Obrazy tła: 9.66%

test
Obrazy: 3509
Etykiety: 2768
Obrazy tła: 21.12%



# Zapisanie przefiltrowanych danych do zip

In [25]:
folder_to_zip = str(data_path)
output_filename = "EAGLE_dataset_sliced_cleaned_obb.zip"

print(f"Pakowanie folderu: {folder_to_zip}...")

# -r = rekursywnie (wszystkie podfoldery)
# -q = cicho (bez wypisywania nazw plików - to przyspiesza i nie zacina Colaba)
!zip -r -q {output_filename} {folder_to_zip}

print(f"Gotowe! Utworzono plik: {output_filename}")

Pakowanie folderu: /content/data/content/EAGLE_Dataset_public_edit_slicing_more...
Gotowe! Utworzono plik: EAGLE_dataset_sliced_cleaned_obb.zip


# Kopiowanie zip na google drive

In [26]:
!cp EAGLE_dataset_sliced_cleaned_obb.zip /content/drive/MyDrive/dane_detekcja_pojazdow

# Utworzenie zbioru danych z obrazami tylko w rozdzielczości 1024

### Tworzenie zbioru danych z obrazami tylko w rozdzielczości 1024x1024

In [27]:
import os
import shutil
from pathlib import Path

# Nowa ścieżka bazowa dla danych o rozdzielczości 1024x1024
data_path_1024_res = Path('/content/data_1024_res')

# Utwórz niezbędne katalogi
for split_folder in ['train', 'val', 'test']:
    (data_path_1024_res / 'images' / split_folder).mkdir(parents=True, exist_ok=True)
    (data_path_1024_res / 'labels' / split_folder).mkdir(parents=True, exist_ok=True)

print(f"Utworzono strukturę katalogów w: {data_path_1024_res}")

target_resolution = 1024

print(f"\nRozpoczynam kopiowanie plików o rozdzielczości {target_resolution}x{target_resolution}...")

# Iteruj przez split-y (train, val, test)
for split_name in ['train', 'val', 'test']:
    print(f"Przetwarzanie splitu: {split_name}")
    original_images_dir = data_path / 'images' / split_name
    original_labels_dir = data_path / 'labels' / split_name

    target_images_dir = data_path_1024_res / 'images' / split_name
    target_labels_dir = data_path_1024_res / 'labels' / split_name

    copied_count = 0

    # Iteruj przez wszystkie pliki obrazów w oryginalnym katalogu
    for img_file in original_images_dir.glob('*'):
        if img_file.is_file():
            stem = img_file.stem
            # Użyj funkcji get_image_dimensions do pobrania wymiarów
            width, height = get_image_dimensions(stem, split_name)

            if width == target_resolution and height == target_resolution:
                # Kopiuj obraz
                shutil.copy(img_file, target_images_dir / img_file.name)

                # Kopiuj odpowiadający plik etykiet, jeśli istnieje
                label_file_name = stem + '.txt'
                original_label_path = original_labels_dir / label_file_name
                if original_label_path.exists():
                    shutil.copy(original_label_path, target_labels_dir / label_file_name)
                copied_count += 1
    print(f"  Skopiowano {copied_count} obrazów z {split_name}.")

print("\nKopiowanie zakończone.")

Utworzono strukturę katalogów w: /content/data_1024_res

Rozpoczynam kopiowanie plików o rozdzielczości 1024x1024...
Przetwarzanie splitu: train
  Skopiowano 4396 obrazów z train.
Przetwarzanie splitu: val
  Skopiowano 1397 obrazów z val.
Przetwarzanie splitu: test
  Skopiowano 3509 obrazów z test.

Kopiowanie zakończone.


### Liczba plików w nowo utworzonym zbiorze danych (1024x1024)

In [28]:
# Wywołaj ponownie funkcję count_files dla nowego zbioru danych
count_files(data_path_1024_res)

val
Obrazy: 1397
Etykiety: 1322
Obrazy tła: 5.37%

train
Obrazy: 4396
Etykiety: 4159
Obrazy tła: 5.39%

test
Obrazy: 3509
Etykiety: 2768
Obrazy tła: 21.12%



# Utworzenie ZIP z przefiltrowanymi danymi (1024x1024)

In [29]:
folder_to_zip_1024 = str(data_path_1024_res)
output_filename_1024 = "EAGLE_dataset_1024_res_cleaned_obb.zip"

print(f"Pakowanie folderu: {folder_to_zip_1024}...")

# -r = rekursywnie (wszystkie podfoldery)
# -q = cicho (bez wypisywania nazw plików - to przyspiesza i nie zacina Colaba)
!zip -r -q {output_filename_1024} {folder_to_zip_1024}

print(f"Gotowe! Utworzono plik: {output_filename_1024}")

Pakowanie folderu: /content/data_1024_res...
Gotowe! Utworzono plik: EAGLE_dataset_1024_res_cleaned_obb.zip


# Kopiowanie ZIP na Google Drive

In [30]:
!cp {output_filename_1024} /content/drive/MyDrive/dane_detekcja_pojazdow

### Tworzenie zbioru danych z obrazami o rozdzielczości 1024x1024 i 2048x2048

In [31]:
import os
import shutil
from pathlib import Path

# Nowa ścieżka bazowa dla danych o rozdzielczości 1024x1024 i 2048x2048
data_path_1024_2048_res = Path('/content/data_1024_2048_res')

# Utwórz niezbędne katalogi
for split_folder in ['train', 'val', 'test']:
    (data_path_1024_2048_res / 'images' / split_folder).mkdir(parents=True, exist_ok=True)
    (data_path_1024_2048_res / 'labels' / split_folder).mkdir(parents=True, exist_ok=True)

print(f"Utworzono strukturę katalogów w: {data_path_1024_2048_res}")

target_resolutions = [1024, 2048]

print(f"\nRozpoczynam kopiowanie plików o rozdzielczościach {target_resolutions}x{target_resolutions}...")

# Iteruj przez split-y (train, val, test)
for split_name in ['train', 'val', 'test']:
    print(f"Przetwarzanie splitu: {split_name}")
    original_images_dir = data_path / 'images' / split_name
    original_labels_dir = data_path / 'labels' / split_name

    target_images_dir = data_path_1024_2048_res / 'images' / split_name
    target_labels_dir = data_path_1024_2048_res / 'labels' / split_name

    copied_count = 0

    # Iteruj przez wszystkie pliki obrazów w oryginalnym katalogu
    for img_file in original_images_dir.glob('*'):
        if img_file.is_file():
            stem = img_file.stem
            # Użyj funkcji get_image_dimensions do pobrania wymiarów
            width, height = get_image_dimensions(stem, split_name)

            if width in target_resolutions and height in target_resolutions:
                # Kopiuj obraz
                shutil.copy(img_file, target_images_dir / img_file.name)

                # Kopiuj odpowiadający plik etykiet, jeśli istnieje
                label_file_name = stem + '.txt'
                original_label_path = original_labels_dir / label_file_name
                if original_label_path.exists():
                    shutil.copy(original_label_path, target_labels_dir / label_file_name)
                copied_count += 1
    print(f"  Skopiowano {copied_count} obrazów z {split_name}.")

print("\nKopiowanie zakończone.")

Utworzono strukturę katalogów w: /content/data_1024_2048_res

Rozpoczynam kopiowanie plików o rozdzielczościach [1024, 2048]x[1024, 2048]...
Przetwarzanie splitu: train
  Skopiowano 5883 obrazów z train.
Przetwarzanie splitu: val
  Skopiowano 1878 obrazów z val.
Przetwarzanie splitu: test
  Skopiowano 3509 obrazów z test.

Kopiowanie zakończone.


### Liczba plików w nowo utworzonym zbiorze danych (1024x1024 i 2048x2048)

In [32]:
# Wywołaj ponownie funkcję count_files dla nowego zbioru danych
count_files(data_path_1024_2048_res)

val
Obrazy: 1878
Etykiety: 1793
Obrazy tła: 4.53%

train
Obrazy: 5883
Etykiety: 5623
Obrazy tła: 4.42%

test
Obrazy: 3509
Etykiety: 2768
Obrazy tła: 21.12%



# Utworzenie ZIP z przefiltrowanymi danymi (1024x1024 i 2048x2048)

In [33]:
folder_to_zip_1024_2048 = str(data_path_1024_2048_res)
output_filename_1024_2048 = "EAGLE_dataset_1024_2048_res_cleaned_obb.zip"

print(f"Pakowanie folderu: {folder_to_zip_1024_2048}...")

# -r = rekursywnie (wszystkie podfoldery)
# -q = cicho (bez wypisywania nazw plików - to przyspiesza i nie zacina Colaba)
!zip -r -q {output_filename_1024_2048} {folder_to_zip_1024_2048}

print(f"Gotowe! Utworzono plik: {output_filename_1024_2048}")

Pakowanie folderu: /content/data_1024_2048_res...
Gotowe! Utworzono plik: EAGLE_dataset_1024_2048_res_cleaned_obb.zip


# Kopiowanie ZIP na Google Drive

In [34]:
!cp {output_filename_1024_2048} /content/drive/MyDrive/dane_detekcja_pojazdow